In [ ]:
import pandas as pd
import numpy as np
import json


In [ ]:
df = pd.read_csv('receipt.csv', sep=',')

df


Готовые отчеты для обработки находятся в папке "Отчет по нейросети"

In [ ]:
#Сроки без ответа нейросети

smp_dict = {
    'started_at': [],
    'uuid': [],
    'response': [],
    'name': [],
    'source': []
    
}

#Берем только завершенные строки, где список skus пустой

for key, row in df[(df.status == 'completed') & (df.skus == '[]')].iterrows():
    
    resp = json.loads(row.response) #превращаю json-строку из response в словрик
    st_dat = row['started_at']
    
    # В одном response может быть несколько товаров, поэтому проходим по каждому
    
    for items in resp['content']['items']:
        
        smp_dict['started_at'].append(st_dat)
        smp_dict['uuid'].append(row.uuid)
        smp_dict['response'].append(resp)
        smp_dict['source'].append(row.loaded)
        
        # у некоторых товаров поле name может отсутствовать
        
        try:
            smp_dict['name'].append(items['name'])
        except KeyError:
            smp_dict['name'].append(None)
            
            
res = pd.DataFrame.from_dict(smp_dict)

# Добавляем пустые поля, чтобы структура отчета совпадала с основным отчетом

res['raw_json'] = None
res['sku (prediction)'] = None
res['errorCode'] = None

cols = [
    'started_at',
    'uuid',
    'response',
    'raw_json',
    'name',
    'sku (prediction)',
    'errorCode',
    'source'
]

res = res[cols]

res.to_excel('./Отчет по нейросети/result_null.xlsx', index = False)



#Сроки c ответом нейросети

mp_dict = {
    'started_at': [],
    'uuid': [],
    'response': [],
    'raw_json': [],
    'name': [],
    'sku (prediction)': [],
    'errorCode': [],
    'source': []
    
} 


# Оставляем завершенные запросы, где skus не пустой и не NaN

for key, row in df[(df.status == 'completed') & (df.skus != '[]') & (df.skus.notnull())].iterrows():
    
    resp = json.loads(row.response) #превращаю json-строку из response в словрик
    st_dat = row['started_at']
    
    try:
        
        pred = json.loads(row.skus) #разбираем предсказания нейросети из JSON
        
        #Одному запросу ...
        
        for items in pred:
            mp_dict['started_at'].append(st_dat)
            mp_dict['uuid'].append(row.uuid)
            mp_dict['response'].append(resp)
            mp_dict['raw_json'].append(pred)
            mp_dict['name'].append(items['name'])
            mp_dict['sku (prediction)'].append(items['SKU'])
            mp_dict['source'].append(row.loaded)
            
            
            #errorCode приходят не во всех ответах
            try:
                mp_dict['errorCode'].append(items['errorCode'])
            except KeyError:
                mp_dict['errorCode'].append(None)
                
                
    #Если значение skus невозможно разобрать как ожидаемый JSON
    
    except TypeError:
        
        mp_dict['started_at'].append(st_dat)
        mp_dict['uuid'].append(row.uuid)
        mp_dict['response'].append(resp)
        mp_dict['raw_json'].append('NaN')
        mp_dict['name'].append('NaN')
        mp_dict['sku (prediction)'].append('NaN')
        mp_dict['source'].append('NaN')
        

res2 = pd.DataFrame.from_dict(mp_dict)
res2.to_excel('./Отчет по нейросети/result_not_null.xlsx', index = False)
        
        
        
        